In [1]:

import functools
import openai

def _default_retries(cls):
    original = cls.__init__

    @functools.wraps(original)
    def __init__(self, *args, **kwargs):
        kwargs.setdefault("max_retries", 12)
        original(self, *args, **kwargs)

    cls.__init__ = __init__

for _cls in (openai.OpenAI, openai.AsyncOpenAI):
    _default_retries(_cls)


## Example Scenario

Imagine it's winter time and you are sitting snugly at your place. You are trying to put together a ski trip with family and friends in 3 days. You are not sure what the weather would be and what's the best way to prepare.

Of course you can Google the weather, but you also want Grok to give you some personalized advices. You set out to build a chatbot with function call that retrieves live weather forecast.

## Overview of Function Call (Tool Call)

> Suggested reading: [Function calling](https://docs.x.ai/developers/tools/function-calling) in the xAI API documentation

Letting Grok use function call involves:
1. Defining a function to perform desired actions on your system.
2. Make the function parameter signature available to Grok in your API request, so Grok knows that these functions are available for use on your local system.
3. When Grok determines that your request requires additional information/action available through those functions, its response includes a `function_call` item with the function name and arguments.
4. Your system runs the function and sends the result back to Grok as a `function_call_output` item.
5. Grok generates response using those results, and might ask for more function calls based on the specific case.

In a nutshell — Grok calls our local function in xAI API response, and our local function returns result to Grok via xAI API request.

## Building Blocks

Now with the basic ideas of function calling, let's set up to build the following:
1. Functions that Grok can use
2. Function call handler to handle when Grok asks for a function call in response
3. Set up the request and response pipeline

First let's install some dependencies:

In [2]:
%pip install --quiet openai pydantic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
XAI_API_KEY = os.getenv("XAI_API_KEY")
MODEL = "grok-4.7"

client = OpenAI(
    base_url="https://api.x.ai/v1",
    api_key=XAI_API_KEY
)

### 1. Creating Functions locally that Grok can use

To fetch the weather forecast, we will use the [NOAA API Web Service](https://www.weather.gov/documentation/services-web-api).

We can get a 7-day weather forecast on a 2.5km grid area in the following format, by retrieving data from 
`https://api.weather.gov/gridpoints/{wfo}/{x},{y}/forecast`. The `wfo` and `{x},{y}` can be obtained from `https://api.weather.gov/points/{latitude in signed decimal degrees},{longitude in signed decimal degrees}`
For example, the following is a weather forecast for Boston, MA, retrieved from endpoint `https://api.weather.gov/gridpoints/BOX/72,90/forecast`
```json
{
    "@context": [
        "https://geojson.org/geojson-ld/geojson-context.jsonld",
        {
            "@version": "1.1",
            "wx": "https://api.weather.gov/ontology#",
            "geo": "http://www.opengis.net/ont/geosparql#",
            "unit": "http://codes.wmo.int/common/unit/",
            "@vocab": "https://api.weather.gov/ontology#"
        }
    ],
    "type": "Feature",
    "geometry": {
        "type": "Polygon",
        "coordinates": [
            [
                [
                    -71.029600000000002,
                    42.345599999999997
                ],
                [
                    -71.0244,
                    42.366999999999997
                ],
                [
                    -71.053399999999996,
                    42.370799999999996
                ],
                [
                    -71.058599999999998,
                    42.349399999999996
                ],
                [
                    -71.029600000000002,
                    42.345599999999997
                ]
            ]
        ]
    },
    "properties": {
        "units": "us",
        "forecastGenerator": "BaselineForecastGenerator",
        "generatedAt": "2025-01-17T18:07:01+00:00",
        "updateTime": "2025-01-17T15:18:01+00:00",
        "validTimes": "2025-01-17T09:00:00+00:00/P8DT6H",
        "elevation": {
            "unitCode": "wmoUnit:m",
            "value": 0.91439999999999999
        },
        "periods": [
            {
                "number": 1,
                "name": "This Afternoon",
                "startTime": "2025-01-17T13:00:00-05:00",
                "endTime": "2025-01-17T18:00:00-05:00",
                "isDaytime": true,
                "temperature": 34,
                "temperatureUnit": "F",
                "temperatureTrend": "",
                "probabilityOfPrecipitation": {
                    "unitCode": "wmoUnit:percent",
                    "value": null
                },
                "windSpeed": "5 to 8 mph",
                "windDirection": "NW",
                "icon": "https://api.weather.gov/icons/land/day/few?size=medium",
                "shortForecast": "Sunny",
                "detailedForecast": "Sunny, with a high near 34. Northwest wind 5 to 8 mph."
            },
            {
                "number": 2,
                "name": "Tonight",
                "startTime": "2025-01-17T18:00:00-05:00",
                "endTime": "2025-01-18T06:00:00-05:00",
                "isDaytime": false,
                "temperature": 28,
                "temperatureUnit": "F",
                "temperatureTrend": "",
                "probabilityOfPrecipitation": {
                    "unitCode": "wmoUnit:percent",
                    "value": null
                },
                "windSpeed": "5 to 10 mph",
                "windDirection": "S",
                "icon": "https://api.weather.gov/icons/land/night/sct?size=medium",
                "shortForecast": "Partly Cloudy",
                "detailedForecast": "Partly cloudy, with a low around 28. South wind 5 to 10 mph."
            },
            {
                "number": 3,
                "name": "Saturday",
                "startTime": "2025-01-18T06:00:00-05:00",
                "endTime": "2025-01-18T18:00:00-05:00",
                "isDaytime": true,
                "temperature": 44,
                "temperatureUnit": "F",
                "temperatureTrend": "",
                "probabilityOfPrecipitation": {
                    "unitCode": "wmoUnit:percent",
                    "value": 70
                },
                "windSpeed": "10 to 14 mph",
                "windDirection": "S",
                "icon": "https://api.weather.gov/icons/land/day/rain,30/rain,70?size=medium",
                "shortForecast": "Light Rain Likely",
                "detailedForecast": "Rain likely after 10am. Cloudy, with a high near 44. South wind 10 to 14 mph. Chance of precipitation is 70%. New rainfall amounts less than a tenth of an inch possible."
            },
            {
                "number": 4,
                "name": "Saturday Night",
                "startTime": "2025-01-18T18:00:00-05:00",
                "endTime": "2025-01-19T06:00:00-05:00",
                "isDaytime": false,
                "temperature": 33,
                "temperatureUnit": "F",
                "temperatureTrend": "",
                "probabilityOfPrecipitation": {
                    "unitCode": "wmoUnit:percent",
                    "value": 60
                },
                "windSpeed": "7 to 10 mph",
                "windDirection": "SW",
                "icon": "https://api.weather.gov/icons/land/night/rain,60/bkn?size=medium",
                "shortForecast": "Light Rain Likely then Mostly Cloudy",
                "detailedForecast": "Rain likely and patchy fog before 11pm. Mostly cloudy, with a low around 33. Southwest wind 7 to 10 mph. Chance of precipitation is 60%. New rainfall amounts less than a tenth of an inch possible."
            },
            // ...
        ]
    }
}
```

For our use case, we will limit the forecast area to a few popular ski areas, and return the `properties.periods` from the API response from NOAA to Grok.

Here, we will define the function inputs and outputs using Pydantic:

In [4]:
from enum import Enum
from typing import Literal
from pydantic import BaseModel, Field
import requests

# Available ski resorts
class SkiResort(str, Enum):
    aspen = 'aspen'
    breckenridge = 'breckenridge'
    jackson_hole = 'jackson_hole'
    vail = 'vail'

# Tool call request available to Grok
class ForecastRequest(BaseModel):
    location: SkiResort = Field(description="Ski resort location name in snake case")


# Probability of precipitation used in response body definition
class ProbabilityOfPrecipitation(BaseModel):
    unitCode: str = Field(description="Unit code of precipitation")
    value: int | None = Field(description="Probability of precipitation in unitCode")

# Response format to send back to Grok
class ForecastResponse(BaseModel):
    number: int = Field(description="Index of the forecast in the sequence")
    name: str = Field(description="Name of the report period, relative to today")
    startTime: str = Field(description="ISO8601 format of forecasting period start with timezone")
    endTime: str = Field(description="ISO8601 format of forecasting period end with timezone")
    isDaytime: bool = Field(description="Whether forecasting period is daytime. True if it is daytime")
    temperature: int = Field(description="Temperature in temperatureUnit unit")
    temperatureUnit: Literal["C", "F"] = Field(description="Temperature Unit")
    temperatureTrend: str = Field(description="Description of temperature trend")
    probabilityOfPrecipitation: ProbabilityOfPrecipitation = Field(description="Probability of Precipitation")
    windSpeed: str = Field(description="Description of Wind Speed")
    windDirection: str = Field(description="Wind direction")
    shortForecast: str = Field(description="A short summary of forecast condition")
    detailedForecast: str = Field(description="Detailed description of the forecast")


# URLs of the forecast locations
skiResortForecastUrl: dict[SkiResort, str] = {
    SkiResort.aspen : "https://api.weather.gov/gridpoints/GJT/156,102/forecast",
    SkiResort.breckenridge : "https://api.weather.gov/gridpoints/BOU/25,53/forecast",
    SkiResort.jackson_hole : "https://api.weather.gov/gridpoints/RIW/42,139/forecast",
    SkiResort.vail : "https://api.weather.gov/gridpoints/GJT/173,121/forecast"
}

# Local function that will be executed when Grok asks for
def get_weather_forecast(**kwargs) -> list[ForecastResponse]:
    req = ForecastRequest(**kwargs)  # Validate and parse the keyword parameters that Grok sends to us
    forecast_url = skiResortForecastUrl[req.location]  # Get request url for a given ski resort location

    forecast = requests.get(url=forecast_url).json()  # Retrieve forecast

    res: list[ForecastResponse] = []

    for item in forecast["properties"]["periods"]:
        item.pop("icon")  # Remove unnecessary weather icon url
        res.append(item)
    return res

You can preview the data we send to Grok, when Grok asks for the weather forecast at Aspen, CO. Here are the first two forecast periods:

In [5]:
get_weather_forecast(location='aspen')[:2]

[{'number': 1,
  'name': 'This Afternoon',
  'startTime': '2026-09-30T13:00:00-06:00',
  'endTime': '2026-09-30T18:00:00-06:00',
  'isDaytime': True,
  'temperature': 60,
  'temperatureUnit': 'F',
  'temperatureTrend': None,
  'probabilityOfPrecipitation': {'unitCode': 'wmoUnit:percent', 'value': 72},
  'windSpeed': '5 mph',
  'windDirection': 'E',
  'shortForecast': 'Showers And Thunderstorms Likely',
  'detailedForecast': 'Showers and thunderstorms likely. Cloudy, with a high near 60. East wind around 5 mph. Chance of precipitation is 70%.'},
 {'number': 2,
  'name': 'Tonight',
  'startTime': '2026-09-30T18:00:00-06:00',
  'endTime': '2026-10-01T06:00:00-06:00',
  'isDaytime': False,
  'temperature': 39,
  'temperatureUnit': 'F',
  'temperatureTrend': None,
  'probabilityOfPrecipitation': {'unitCode': 'wmoUnit:percent', 'value': 20},
  'windSpeed': '0 to 5 mph',
  'windDirection': 'SE',
  'shortForecast': 'Slight Chance Showers And Thunderstorms then Mostly Cloudy',
  'detailedForeca

The function to call is defined! Hurray! Now we need to send the function name and parameters signature so that Grok knows how to call the function.

In [6]:
# Definition of parameters with Pydantic JSON schema
tools_definition = [
    {
        "type": "function",
        "name": "get_weather_forecast",  # the function name that we defined
        "description": "Get the weather forecast at a given location",  # Description of the function, so that Grok knows whether the function would be useful to solving the problem
        "parameters": ForecastRequest.model_json_schema(),  # Generate the request parameter schema from Pydantic
    },
]

### 2. Function handler to invoke the function we defined and send the result back

With our previous definition of the function, we can send a request to Grok.

Let's see how Grok will respond:

In [7]:
question = "What should I prepare for a ski trip on Wednesday according to the weather in Vail, CO?"

response = client.responses.create(
    model=MODEL,
    input=[{"role": "user", "content": question}],
    tools=tools_definition,  # The list of our functions and their parameters
)

# Instead of an answer, the response contains a function call
[item for item in response.output if item.type == "function_call"]

[ResponseFunctionToolCall(arguments='{"location":"vail"}', call_id='call-7e1be1bd-d0d3-43a7-935b-67ce07e1864c-0', name='get_weather_forecast', type='function_call', id='fc_3ff89094-49a1-9481-9d14-5638d33277bc_0', async_=None, caller=None, namespace=None, status='completed')]

In `response.output`, Grok included a `function_call` item with the function's `name`, its `arguments` as a JSON string, and a `call_id`, for example `name='get_weather_forecast', arguments='{"location":"vail"}'`.

We need a handler that:
1. Finds every `function_call` item in Grok's response.
2. Calls the matching local function with the arguments Grok chose.
3. Packages each result as a `function_call_output` item with the same `call_id`, so Grok can tell which call the result belongs to:
    ```json
    {
        "type": "function_call_output",
        "call_id": "call_67298806", // The id Grok gave the function call
        "output": "[...]" // The JSON string returned by get_weather_forecast()
    }
    ```

We don't need to send the whole conversation again. Passing `previous_response_id` tells the API to continue from Grok's earlier response, which already has the question and the function call.

In [8]:
import json
from typing import Callable

# Define a mapping between function name and the function's callable object
tools_map: dict[str, Callable] = {
    "get_weather_forecast": get_weather_forecast
}

def run_function_calls(response) -> list[dict]:
    """Run every function call in Grok's response and return the results as function_call_output items."""
    outputs = []
    for item in response.output:
        if item.type != "function_call":
            continue

        # Get the function name and arguments Grok wants to call
        function_args = json.loads(item.arguments)

        # Call the matching function defined earlier in tools_map
        result = tools_map[item.name](**function_args)

        outputs.append(
            {
                "type": "function_call_output",
                "call_id": item.call_id,  # call_id supplied in Grok's response
                "output": json.dumps(result),
            }
        )
    return outputs

Now we run the handler on Grok's earlier response and look at what we'll send back:

In [9]:
function_outputs = run_function_calls(response)

# Preview each result, since the full forecast is long
for item in function_outputs:
    print(item["call_id"], item["output"][:300], "...")

call-7e1be1bd-d0d3-43a7-935b-67ce07e1864c-0 [{"number": 1, "name": "This Afternoon", "startTime": "2026-09-30T13:00:00-06:00", "endTime": "2026-09-30T18:00:00-06:00", "isDaytime": true, "temperature": 58, "temperatureUnit": "F", "temperatureTrend": null, "probabilityOfPrecipitation": {"unitCode": "wmoUnit:percent", "value": 42}, "windSpeed":  ...


**Now we can finally send this back to Grok to get the recommendation!**

In [10]:
final_response = client.responses.create(
    model=MODEL,
    input=function_outputs,
    tools=tools_definition,
    previous_response_id=response.id,  # Continue from the response that asked for the function call
)

print(final_response.output_text)

**Wednesday in Vail looks mild and wetter than typical ski weather**—not cold enough for reliable snow.

From the forecast (Wednesday afternoon into tonight):

- **Afternoon:** Chance of showers and thunderstorms (about 40%), cloudy, high near **58°F**, light east wind around 5 mph.
- **Tonight:** Slight chance of rain showers early, then mostly cloudy, low around **35°F**, light east wind.

### What to prepare
- **Rain shell / waterproof jacket and pants** — showers and possible thunderstorms are the main issue.
- **Layers, not full winter kit** — daytime is mild (upper 50s); pack a light insulating mid-layer and something warmer for the evening drop toward the mid-30s.
- **Moisture-wicking base layers** and a change of dry clothes in case you get wet.
- **Light gloves and a warm hat** for the cooler night, but heavy mitts and extreme-cold gear are unnecessary based on these temps.
- **Good traction footwear** if you’ll be walking around; don’t count on fresh snow.
- **Sun protection*

### 3. Setting up the whole pipeline to generate recommendation

As a recap:
- We have defined our local function `get_weather_forecast()` that retrieves NOAA weather forecast for Grok to run.
- We have defined `run_function_calls()` to run the functions Grok asks for and package the results as `function_call_output` items.
  - As part of this, we also defined a `tools_map` to map function name given by Grok -> function Callable object.

Grok can ask for more than one round of function calls, so the pipeline keeps sending results back until a response has none. It also caps the number of rounds, so a model that keeps calling functions can't loop forever:

In [11]:
def ask_grok(question: str, max_rounds: int = 5) -> str:
    response = client.responses.create(
        model=MODEL,
        input=[{"role": "user", "content": question}],
        tools=tools_definition,
    )
    for _ in range(max_rounds):
        function_outputs = run_function_calls(response)
        if not function_outputs:
            return response.output_text  # No more function calls, so this is Grok's answer
        response = client.responses.create(
            model=MODEL,
            input=function_outputs,
            tools=tools_definition,
            previous_response_id=response.id,
        )
    raise RuntimeError(f"Grok was still calling functions after {max_rounds} rounds")


print(ask_grok("Which of Aspen, Breckenridge, or Vail will have the best skiing conditions this weekend?"))

**None of them will have good skiing conditions this weekend — it’s far too warm, with no snow in the forecast.**

Here’s the weekend outlook (Sat–Sun):

| Resort | Saturday | Sunday | Notes |
|--------|----------|--------|-------|
| **Aspen** | Mostly sunny, high ~76°F, low ~45°F, 0% precip | Sunny, high ~77°F, low ~45°F, ~1% precip | Warmest |
| **Breckenridge** | Sunny, high ~69°F, low ~38°F, 0% precip | Sunny, high ~69°F, low ~38°F, ~3% precip | Coolest |
| **Vail** | Sunny, high ~72°F, low ~40°F, 0% precip | Sunny, high ~73°F, low ~42°F, ~1% precip | In between |

Breckenridge is the coolest of the three, but highs in the upper 60s and overnight lows only in the upper 30s mean no natural snow and poor (or nonexistent) ski conditions at all three resorts. This is early October, well before typical ski season.


Try `ask_grok()` with your own questions. To hold a longer conversation, keep passing `previous_response_id` from one request to the next so Grok remembers what came before.